# Portfolio Tracker — Guía de uso con PortfolioClient

Este notebook muestra cómo usar `PortfolioClient`, la fachada Python que unifica Portfolio + DataProviders + TaxOptimizer.
Todos los métodos devuelven `pd.DataFrame`, ideal para exploración interactiva.

## 1. Inicializar PortfolioClient

Se carga desde el Excel de órdenes. El sistema FIFO se aplica automáticamente.

In [ ]:
import sys, os
import pandas as pd
from IPython.display import display

# Resolver rutas
sys.path.append(os.path.abspath(".."))
data_file = "../data/Órdenes 1238478.tsv"
cache = "../data/cache"

from app.client import PortfolioClient

client = PortfolioClient(source=data_file, cache_path=cache)
print(f"✅ PortfolioClient inicializado — {len(client.portfolio.positions)} fondos, "
      f"{len(client.portfolio.open_lots)} lotes abiertos")

✅ PortfolioClient inicializado — 18 fondos, 59 lotes abiertos


## 2. Posiciones con P&L en vivo

`client.positions(live=True)` obtiene precios actuales y calcula ganancia/pérdida por fondo.

In [ ]:
df_pos = client.positions(live=True)
display(df_pos)
print(f"\nCapital invertido: {client.portfolio.get_total_invested():,.2f} €")
print(f"Valor actual:      {df_pos['Valor_Actual'].sum():,.2f} €" if df_pos['Valor_Actual'].notna().any() else "")

,ISIN,Fondo,Valor_Actual,Capital_Invertido,Ganancia_Euros,Ganancia_Pct,Participaciones,Precio_Actual,Fecha_Actualizacion
0,IE00BYX5MX67,IE00BYX5MX67,23621.10,25002.88,-1381.78,-5.53,1849.299,12.773000,2026-05-02 19:09
1,IE00BD0NCM55,IE00BD0NCM55,18218.88,16402.45,1816.43,11.07,689.900,26.408001,2026-05-02 19:09
2,IE00BYX5NX33,IE00BYX5NX33,17473.35,16141.22,1332.13,8.25,1332.135,13.116800,2026-05-02 19:09
3,LU0302296495,LU0302296495,5211893.88,11000.00,5200893.88,47280.85,3122.018,1669.399048,2026-05-02 19:09
4,LU0273159177,LU0273159177,6365.70,7019.02,-653.32,-9.31,20.925,304.214996,2026-05-02 19:09
5,IE00BH6XSF26,IE00BH6XSF26,6682.66,6509.51,173.15,2.66,21.411,312.113586,2026-05-02 19:09
6,LU1598719752,LU1598719752,5767.35,5500.00,267.35,4.86,32.883,175.389999,2026-05-02 19:09
7,LU0840158819,LU0840158819,5065.73,5000.00,65.73,1.31,32.274,156.960007,2026-05-02 19:09
8,LU2466448532,LU2466448532,5482.07,5000.00,482.07,9.64,28.184,194.509995,2026-05-02 19:09
9,LU0329355670,LU0329355670,5829.08,5000.00,829.08,16.58,15.899,366.631989,2026-05-02 19:09



Capital invertido: 122,528.28 €
Valor actual:      5,336,295.25 €


# Movimientos

In [ ]:
# Todos los movimientos
df_mov = client.movements("LU0302296495")
df_mov = client.movements()
display(df_mov)

# Filtrar por ISIN concreto
# df_mov = client.movements(isin="LU0273159177")
# display(df_mov)

# Filtrar por nombre (contains, case-insensitive)
# df_mov = client.movements(name="amundi")
# display(df_mov)


,Fecha,ISIN,Importe,Participaciones,Estado
0,2024-10-27,FR0000989626,2500.0,59.000,Finalizada
1,2024-12-02,IE00BYX5MX67,2000.0,142.707,Finalizada
2,2024-12-14,IE00BYX5MX67,500.0,35.484,Finalizada
3,2024-12-26,IE00BYX5MX67,1000.0,70.730,Finalizada
4,2025-01-13,IE00BYX5MX67,1000.0,71.645,Finalizada
...,...,...,...,...,...
60,2026-03-24,LU1598719752,500.0,2.977,Finalizada
61,2026-03-24,IE00BH6XSF26,500.0,1.656,Finalizada
62,2026-03-24,IE000ZYRH0Q7,1000.0,96.140,Finalizada
63,2026-04-15,LU3038481936,1500.0,5.317,Finalizada


In [ ]:
df_mov.loc[lambda x: x.ISIN.str.contains("LU0302296495")]

,Fecha,ISIN,Importe,Participaciones,Estado
13,2025-08-16,LU0302296495,2000.0,1.405,Finalizada
19,2025-09-11,LU0302296495,2000.0,1.343,Finalizada
23,2025-09-23,LU0302296495,1000.0,0.650,Finalizada
24,2025-09-23,LU0302296495,1000.0,651.000,Finalizada
29,2025-10-07,LU0302296495,1000.0,628.000,Finalizada
33,2025-10-21,LU0302296495,1000.0,619.000,Finalizada
35,2025-10-30,LU0302296495,1000.0,598.000,Finalizada
37,2025-11-06,LU0302296495,1000.0,622.000,Finalizada
43,2026-01-16,LU0302296495,1000.0,0.620,Finalizada


## 3. Enriquecer datos de fondos

`client.enrich()` recorre todos los ISINs del portfolio, consultando FMP → Morningstar → Yahoo Finance para obtener expense ratios, ratings, exposición geográfica, etc.

In [ ]:
df_enriched = client.enrich()
display(df_enriched)

# Resumen por categoría
df_summary = client.summary()
print("\n--- Resumen por tipo de activo ---")
display(df_summary)

# Detalle de un fondo concreto (primer ISIN del portfolio)
isin_ejemplo = list(client.portfolio.positions.keys())[0]
df_detail = client.fund_details(isin_ejemplo)
print(f"\n--- Detalle de {isin_ejemplo} ---")
display(df_detail)

Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache mana

,ISIN,Nombre,Precio_Actual,Expense_Ratio,AUM,Inception_Date,Rating_MS,Risk_Score,Renta_Variable_Pct,Renta_Fija_Pct,EEUU_Pct,Europa_Pct,Source
0,ES0140794001,Gamma Global A FI,13.778340,None,None,None,None,None,None,None,0,0,Morningstar
1,ES0141116030,ES0141116030,0.000000,None,None,None,None,None,None,None,0,0,Morningstar
2,ES0146309002,Horos Value Internacional FI,215.108963,None,None,None,None,None,None,None,0,0,Morningstar
3,IE000ZYRH0Q7,iShares Dev Wld Idx (IE) S Acc EUR,11.203000,None,None,None,None,None,None,None,0,0,Morningstar
4,IE00B88WFS66,Federated Hermes Asia ex-Japan Equity Fund Cla...,8.653000,None,None,None,None,None,None,None,0,0,Morningstar
5,IE00BD0NCM55,iShares Dev Wld Idx (IE) D Acc EUR,26.266001,None,None,None,None,None,None,None,0,0,Morningstar
6,IE00BH6XSF26,Heptagon Kopernik Glb AllCp Eq AE € Acc,312.118195,None,None,None,None,None,None,None,0,0,Morningstar
7,IE00BYX5MX67,Fidelity Ucits II Icav - Fidelity S&P 500 Inde...,12.773000,None,None,None,None,None,None,None,0,0,Morningstar
8,IE00BYX5NX33,Fidelity MSCI World Index EUR P Acc,13.045500,None,None,None,None,None,None,None,0,0,Morningstar
9,LU0273159177,Deutsche Invest I - Deutsche Invest I Gold and...,302.402008,None,None,None,None,None,None,None,0,0,Morningstar


Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BYX5MX67 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BD0NCM55 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BYX5NX33 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU0302296495 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU0273159177 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00BH6XSF26 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU1598719752 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU0840158819 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para LU2466448532 (detailed)
C

,Tipo,Num_Fondos,Capital_Invertido,Valor_Actual,Peso_Pct
0,Renta Variable,13,104556.06,107432.84,79.50
1,Alternativo,2,7019.02,18056.36,13.36
2,Renta Fija,3,9500.00,9642.08,7.14


Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)

--- Detalle de ES0140794001 ---


...

: 

: 

: 

: 

: 

## 4. Optimización fiscal — retirar 50.000€

`TaxOptimizer` selecciona los lotes con menor ganancia patrimonial para minimizar el impacto fiscal bajo los tramos españoles 2024.

In [ ]:
df_tax = client.tax_optimize(50_000)
display(df_tax)

# Métricas globales del plan
print(f"\nImpuestos estimados: {df_tax.attrs['estimated_tax']:,.2f} €")
print(f"Cantidad neta:       {df_tax.attrs['net_amount']:,.2f} €")

Cache manager initialized with base path: c:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\app\data\cache

--- Procesando: ES0141116030 (light)---
✗ Error obteniendo datos para ES0141116030: Invalid ISIN number: ES0141116030
  [Error] obteniendo NAV ligero para ES0141116030: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.


,ISIN,Fondo,Fecha_Compra,Participaciones_Vendidas,Importe_Retirado,Ganancia_Patrimonial
0,IE00BYX5MX67,IE00BYX5MX67,2024-12-26 00:00:00,38.675000,493.995766,-52.801915
1,LU0273159177,LU0273159177,2026-01-29 00:00:00,3.162000,956.195149,-43.804851
2,LU0273159177,LU0273159177,2026-01-30 00:00:00,6.324000,1912.390299,-87.609701
3,LU0273159177,LU0273159177,2026-01-31 00:00:00,6.059000,1832.253767,-167.746233
4,LU0273159177,LU0273159177,2026-02-25 00:00:00,5.380000,1626.922803,-392.097197
5,LU3038481936,LU3038481936,2026-02-24 00:00:00,6.942000,1955.089264,-44.910736
6,IE00BH6XSF26,IE00BH6XSF26,2026-02-25 00:00:00,3.171000,989.726795,-19.783205
7,LU3038481936,LU3038481936,2026-04-15 00:00:00,5.317000,1497.437283,-2.562717
8,IE00BYX5MX67,IE00BYX5MX67,2025-01-04 00:00:00,78.558000,1003.421315,3.421315
9,IE00BYX5MX67,IE00BYX5MX67,2025-01-13 00:00:00,71.645000,915.121568,-84.878432



Impuestos estimados: 0.00 €
Cantidad neta:       50,000.00 €


: 

: 

: 

: 

: 

## 5. Performance y correlación

Métricas cuantitativas del portfolio (Sharpe, Sortino, Max Drawdown) y matriz de correlación entre fondos.

In [ ]:
# Métricas de performance (3 años)
df_perf = client.performance(years=3)
display(df_perf)

# Matriz de correlación
df_corr = client.correlation(years=3)
if not df_corr.empty:
    print("\n--- Correlación entre fondos ---")
    display(df_corr.style.background_gradient(cmap="RdYlGn", vmin=-1, vmax=1))
else:
    print("Datos insuficientes para calcular correlación.")

Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0146309002 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE000ZYRH0Q7 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE000ZYRH0Q7 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00B88WFS66 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00B88WFS66 (detailed)
Cache manager in

,Metric,Value


Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0140794001 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0141116030 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para ES0146309002 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE000ZYRH0Q7 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE000ZYRH0Q7 (detailed)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00B88WFS66 (light)
Cache manager initialized with base path: ../data/cache
  ✓ Usando datos en caché para IE00B88WFS66 (detailed)
Cache manager in

,Gamma Global A FI,Horos Value Internacional FI,iShares Dev Wld Idx (IE) S Acc EUR,Federated Hermes Asia ex-Japan Equity Fund Class R EUR Accumulating,iShares Dev Wld Idx (IE) D Acc EUR,Heptagon Kopernik Glb AllCp Eq AE € Acc,Fidelity Ucits II Icav - Fidelity S&P 500 Index Fund,Fidelity MSCI World Index EUR P Acc,Deutsche Invest I - Deutsche Invest I Gold and Precious Metals Equities,DNB Fund - Technology,Robeco Capital Growth Funds - Robeco QI Emerging Markets Active Equities,Storm Bond RC EUR,Cobas International P Acc EUR,DNCA Invest Alpha Bonds,Buy & Hold Luxembourg B&H Bond,Echiquier Space,LU3038481936
Gamma Global A FI,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Horos Value Internacional FI,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
iShares Dev Wld Idx (IE) S Acc EUR,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Federated Hermes Asia ex-Japan Equity Fund Class R EUR Accumulating,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
iShares Dev Wld Idx (IE) D Acc EUR,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Heptagon Kopernik Glb AllCp Eq AE € Acc,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Fidelity Ucits II Icav - Fidelity S&P 500 Index Fund,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Fidelity MSCI World Index EUR P Acc,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Deutsche Invest I - Deutsche Invest I Gold and Precious Metals Equities,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
DNB Fund - Technology,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


: 

: 

: 

: 

: 